# Pose completion: our models against the others, on their Pareto fronts








The completions of PoseD-Flow's protocol (four occlusions, K = 10 hypotheses per pose), one split at a time (a
selector above the plots):

- **validation**: every setting tried so far on the fixed validation poses, by the validation scripts of
  `explore/evaluation/completion_diagnostics` and the Optuna studies of `scripts/tune_completion.py`; settings are
  chosen here;
- **test**: the rows of the completion cache on the 5000 test poses (`scripts/evaluate_completion.py` and the test
  scripts of the diagnostics), each a setting chosen on validation and run once; only to report, never to choose.

Both are read in one form by `hp.evaluation.completion_results` (`load`, `load_test`); the two splits are never drawn
together.

**The question: is RSGM (ours) as good as DPoser and PoseD-Flow?** Each family of models has a colour; its front is
drawn through the best of all its settings, whatever the network and the completion method (the marker shows the
method: inpainting, D-Flow, DPoser's optimization). One family is better than another where its front lies on the
better side (left and up).

A single number does not rank the settings: the hypotheses trade accuracy for diversity. The mean MPVPE of K
hypotheses is lowest when they collapse onto one pose, the best of K and the APD (the mean distance between the
hypotheses of a pose) grow with their spread. Hence two plots, both against the diversity:

- **mean ↓ against APD ↑**: accuracy against diversity. APD alone is not a goal (random poses have the largest APD);
  the dotted line is the APD of D-Flow (Heun EDM 50, RSGM small std c10) on the 1000 poses, the floor of the test rule
  of the inpainting studies.
- **best of 10 ↓ against APD ↑**: whether the spread covers the plausible completions.

Only results on the same poses compare: the first 100 validation poses (the smoke tests on the Mac) are not the 1000 of
the studies, and neither are the 5000 test poses.

In [ ]:
import html
import io
import sys
from pathlib import Path

# the code of the checkout this notebook is in (e.g. a worktree), before an editable install of another one
SRC = next(str(p / "src") for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hp").is_dir())
if SRC not in sys.path:
    sys.path.insert(0, SRC)
SCRIPTS = str(Path(SRC).parent / "scripts")  # select_test_points.py, for the save of the test points
if SCRIPTS not in sys.path:
    sys.path.append(SCRIPTS)

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import optuna
from IPython.display import display

from hp.evaluation import completion_results as R
from hp.evaluation import fronts as F
from hp.evaluation.completion import OCCLUSIONS
from hp.visualization import fronts as V

assert sys.modules["hp.evaluation.fronts"].__file__.startswith(SRC), \
    f"hp comes from {sys.modules['hp.evaluation.fronts'].__file__}: restart the kernel"

# ---- parameters ----------------------------------------------------------------------
RESULTS = Path("~/data/experiments/completion").expanduser()  # the .jsonl of the scripts, optuna/*.db of the studies
SOURCES = sorted(RESULTS.glob("*.jsonl")) + sorted((RESULTS / "optuna").glob("*.db"))  # copies from the servers too
CACHE = RESULTS / "cache"  # the test rows (cache/<label>/<occlusion>.pt and _metrics.pt)
SPLIT = "validation"  # the split shown at first: "validation" or "test"
POSES = 1000  # the validation poses shown at first (a selector offers every number in the results)

E = "~/data/experiments/"
NAMES = {  # the run folders of the studies: the names of dflow_valid.MODELS
    E + "rsgm_posedflow_small_s10/run_0001": "small std c10",
    E + "rsgm_posedflow_deep_s10/run_0001": "deep",
    E + "rsgm_posedflow_abl_residual_s10/run_0001": "abl residual",
    E + "rsgm_posedflow_abl_norm_s10/run_0001": "abl norm",
    E + "rsgm_posedflow_res_norm_s10/run_0001": "res norm",
    E + "full/rsgm_posedflow_dposer_like_s10/run_0001": "dposer like full",
    E + "full/rsgm_posedflow_dposer_like_z_s10/run_0001": "dposer like z full",
    E + "full/rsgm_posedflow_small_s10/run_0001": "small std full",
    E + "full/rsgm_posedflow_small_lik_s10/run_0001": "small lik full",
    E + "full/rsgm_posedflow_search_lik_s10/run_0001": "search lik full",
    E + "dposer_posedflow_rsgm_budget_s10/run_0001": "DPoser c10",
    E + "full/dposer_posedflow_s10/run_0001": "DPoser full",
    E + "full/rsgm_posedflow_dposer_like_aniso_s10/run_0001": "dposer like aniso full",
    E + "igso_speed/rsgm_posedflow_abl_norm_aniso_s10/run_0001": "abl norm aniso",
    "~/data/posedflow_ckpt/checkpoints/epoch-009_step-0_loss-0.0000.ckpt": "PoseD-Flow: their checkpoint",
}
# the families of models and their colours (those of pareto_posedflow_vs_rsgm.ipynb); a model is of the first family
# whose prefix its name starts with, else RSGM
FAMILIES = {"RSGM (ours)": "#e34948", "DPoser": "#7b3fa0", "PoseD-Flow": "#2a78d6"}
PREFIXES = {"DPoser": "DPoser", "PoseD-Flow": "PoseD-Flow", "posedflow": "PoseD-Flow"}
METRICS = {  # key of a record: (axis label, direction)
    "mean": ("MPVPE mean (mm)", "min"),
    "best_of_k": ("MPVPE best of 10 (mm)", "min"),
    "apd": ("APD (cm)", "max"),
}
PANELS = [("apd", "mean"), ("apd", "best_of_k")]  # (x, y) of each plot: APD across, as in plot_study_fronts.py
APD_FLOOR = 6.40  # APD of D-Flow (Heun EDM 50, small std c10) on the 1000 validation poses: the test rule's floor
#                  (drawn on validation only)
MARKERS = {"inpaint": "o", "dflow": "*", "dposer": "s"}  # per method
SHORT = {"inpaint": "inpaint", "dflow": "D-Flow", "dposer": "DPoser opt."}  # per method, in the labels of the points
LIMITS = {"mean": 115, "best_of_k": 65}  # the view ends here (mm): settings beyond are counted, not drawn (the fronts
#                                          are of all of them; the degenerate ones, near random poses, go far beyond)
GROUPS = "family x method"  # at first a front per "family" (whatever the method) or per "family x method"
GROUP_COLORS = {  # the colours of the fronts per family x method (per family: FAMILIES)
    ("DPoser", "dposer"): "tab:orange", ("DPoser", "inpaint"): "tab:red", ("PoseD-Flow", "dflow"): "tab:purple",
    ("RSGM (ours)", "inpaint"): "tab:blue", ("RSGM (ours)", "dflow"): "tab:green", ("RSGM (ours)", "dposer"): "tab:cyan",
}
OURS = "RSGM (ours)"  # the family whose fronts are solid; the others are dashed
CORNER = "x"  # the staircase of a front (V.draw_front), as in plot_study_fronts.py
LEGEND = "upper left"  # where the legend of the first plot goes
FRONT_STYLE = "steps"  # "steps", "lines" or "points" (V.STYLES)
MODEL_FRONTS = False  # at first also a thin front for every model, besides the front of each family
LABELS = False  # at first labels on the points of the family fronts
POINTS = True  # at first the points of all experiments (False: only the fronts and their experiments)
POINTS_FILE = Path(SRC).parent / "config" / "completion_test_points.yaml"  # the points of the rule (on the test split)
TEST_POINTS = 10  # at first the test points per front (crosses; 0: none), the rule of scripts/select_test_points.py
FIGURES = RESULTS / "figures"  # the default folder of the save button
# --------------------------------------------------------------------------------------

## 1. The results

Every record is one setting of one completion method on one model. The same setting may be in a script and in a
study: both are kept, they are separate runs.

In [ ]:
def family_of(model):
    return next((f for prefix, f in PREFIXES.items() if model.startswith(prefix)), "RSGM (ours)")


records = [{**r, "split": "validation"} for r in R.load(SOURCES)] + R.load_test(CACHE)
for r in records:
    r["model"] = NAMES.get(r["model"], r["model"])
    r["family"] = family_of(r["model"])
MODELS = sorted({r["model"] for r in records}, key=lambda m: (list(FAMILIES).index(family_of(m)), m))

counts = {}
for r in records:
    counts.setdefault((r["split"], r["poses"], r["family"], r["model"], r["method"]), []).append(r)
print(f"{len(records)} records from {len(SOURCES)} files and the test cache {CACHE}")
for (split, poses, family, model, method), rows in sorted(
        counts.items(), key=lambda kv: (kv[0][0] == "test", kv[0][1], list(FAMILIES).index(kv[0][2]), kv[0][3:])):
    sources = ", ".join(sorted({r["source"].split(":")[0] for r in rows}))
    print(f"  {split:10s} {poses:5d} poses  {family:12s} {model:40s} {R.METHODS[method].split(' (')[0]:22s} "
          f"{len(rows):4d}  ({sources})")

## 1a. What the studies varied

Every Optuna study (`config/completion_tuning/<study>.yaml`, `scripts/tune_completion.py`) fixes a model and a method
and searches the settings of the method on the same 1000 validation poses with the same seeds; the table below is read
from the configurations the studies recorded. The knobs of each method, and which way they move a setting along the
front (from accurate and alike hypotheses to diverse ones):

- **inpainting** (`hp.models.inpainting`; RSGM and DPoser): `steps` of the reverse process, `guidance` ζ (the
  gradient of the error on the visible joints, DPS-like: larger is closer to the observation, less diverse),
  `resample` r (RePaint: r passes per step, more consistent hidden joints), `replace` (the visible joints replaced by
  the noised observation at every step), `schedule` (fixed, EDM).
- **D-Flow** (`hp.models.dflow`; PoseD-Flow, RSGM): `lr` and `steps` of the optimizer of the start of the ODE
  (fewer or smaller steps: closer to samples of the prior, more diverse), `running_weight` (their running cost, keeps
  the trajectory near plausible rotations); fixed: the ODE (PoseD-Flow their Euler with 100 steps, RSGM Heun EDM with
  25 steps, 50 network calls), `invert` (off), `guess_weight` w of the start sqrt(w) guess + sqrt(1 - w) noise
  (theirs 0.25, fixed in all but `posedflow_dflow_guess_mean_apd`, where lr, steps, `running_weight` are fixed;
  smaller w: more diverse).
- **DPoser's method** (`hp.models.score_completion`; DPoser, and on SO(3) for RSGM as score distillation): `lr`,
  the window of noise levels `t_max`, `t_min = t_max * t_min_fraction` (wider and higher: more diverse),
  `iterations` × `steps_per_iter` (DPoser), `prior_weight` (RSGM), the start (`noise`) and `scale` (RSGM, fixed).

The other records (the `.jsonl` of the validation scripts) are single settings, not searches; the studies made before
the configuration was recorded (`completion_<method>_<run>_v<poses>`) are not listed.

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def searched(spec):
    if isinstance(spec, list):
        return "{" + ", ".join(map(str, spec)) + "}"
    return f"[{spec['low']:g}, {spec['high']:g}]" + (" log" if spec.get("log") else "")


print(f"{'study':46s} {'model':28s} {'method':12s} {'objective':9s} {'trials':>6s}  searched; fixed")
for path in (p for p in SOURCES if p.suffix == ".db"):
    for summary in optuna.get_all_study_summaries(f"sqlite:///{path}"):
        study = optuna.load_study(study_name=summary.study_name, storage=f"sqlite:///{path}")
        config = study.user_attrs.get("config")
        if not config:  # older studies, before the configuration was recorded
            continue
        done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
        knobs = ", ".join(f"{k} {searched(v)}" for k, v in config["params"].items())
        fixed = ", ".join(f"{k}={v}" for k, v in (config.get("fixed") or {}).items())
        print(f"{summary.study_name[:46]:46s} {NAMES.get(config['model'], config['model'])[:28]:28s} "
              f"{SHORT[config['method']]:12s} {config['objective']:9s} {done:6d}  {knobs}" + (f"; {fixed}" if fixed else ""))

## 2. The fronts of each family, in numbers

The settings on the front of each family (mean against APD, the first plot), from the most accurate to the most
diverse, with the model, the method and all three metrics. The selectors of section 3 change the poses, the occlusion
and the models of this table too.

In [ ]:
def at(rows, occlusion):
    """The records with the metrics of one occlusion (None: the means over the occlusions)."""
    return [R.at_occlusion(r, occlusion) for r in rows]


def short(options):
    return ", ".join(f"{k}={v:.3g}" if isinstance(v, float) else f"{k}={v}" for k, v in sorted(options.items()))


def objectives(x, y):
    return (x, METRICS[x][1]), (y, METRICS[y][1])


def groups_of(rows, by):
    """The groups of records with a front each, in the order of FAMILIES then MARKERS: {name: (family, method or None,
    records)}; by "family" or "family x method"."""
    out = {}
    for family in FAMILIES:
        for method in ([None] if by == "family" else list(MARKERS)):
            mine = [r for r in rows if r["family"] == family and method in (None, r["method"])]
            if mine:
                out[family if method is None else f"{family} + {SHORT[method]}"] = (family, method, mine)
    return out


def table(rows, x="mean", y="apd", by="family"):
    out = io.StringIO()
    for name, (family, method, mine) in groups_of(rows, by).items():
        print(f"\n{name}: {len(mine)} experiments of {len({r['model'] for r in mine})} models; front of {x} and {y}:",
              file=out)
        print(f"  {'mean':>7s} {'best10':>7s} {'APD':>6s}  {'model':30s} {'method':12s} setting", file=out)
        for r in F.front(mine, *objectives(x, y)):
            print(f"  {r['mean']:7.2f} {r['best_of_k']:7.2f} {r['apd']:6.2f}  {r['model'][:30]:30s} "
                  f"{SHORT[r['method']]:12s} {r['setting']} ({r['source'].split(':')[-1]}) {short(r['options'])}",
                  file=out)
    return out.getvalue()


print(table(at([r for r in records if r["split"] == SPLIT and r["poses"] == POSES], None), by=GROUPS))

## 3. The plots

Each family in its colour: the points of all its settings (faint; the marker is the method) and its front (solid,
through its best settings). Optional: a thin dashed front for every model, and labels on the points of the family
fronts. Above the plots: the split (validation or test) and its poses, the occlusion (or the mean over the four), the models drawn, the style
of the fronts, and the save button (the format by the extension: .png, .pdf, .svg). A click on a point (with
ipyevents, e.g. JupyterLab) highlights its model.

**Test points** (validation only): the crosses are the points of each front that the rule of the test takes
(`scripts/select_test_points.py`, `hp.evaluation.fronts.spanning`): for each of N thresholds of APD evenly spaced from
4.0 to 8.5 cm, the lowest mean among the trials of the configured Optuna studies with APD at least that; listed under
the plots. The rule of the test is N = 10 from 4.0 to 8.5 cm by the lowest mean, per family x method, the mean of the four
occlusions, all models; the controls (N, the APD range, the metric) and other choices here only explore it, and the
list under the plots says when they differ from it.

In [ ]:
def arrow(metric, axis):
    """The way to better on an axis: lower for MPVPE, higher for APD."""
    lower = METRICS[metric][1] == "min"
    return ("← better" if lower else "better →") if axis == "x" else ("better ↓" if lower else "better ↑")


def test_points(groups, rule, split):
    """{group: [(record, thresholds)]}: the points the rule of the test (scripts/select_test_points.py,
    hp.evaluation.fronts.spanning) takes from each group, among the trials of configured studies; validation only.
    rule: (n, APD min, APD max, metric)."""
    n, low, high, metric = rule
    if split == "test":
        return test_rows_of_points(groups)
    if n < 1 or high < low:
        return {}
    return {name: F.spanning(candidates_of(mine), n, low, high, y=metric) for name, (_, _, mine) in groups.items()}


def test_rows_of_points(groups):
    """On the test split: the rows of the points of the rule (POINTS_FILE), with their thresholds, in each group, in
    the order of the points in the file (their APD on validation), so that a point keeps its number of validation; a
    test row is matched by its cache folder (the label with / and spaces as _)."""
    import yaml
    if not POINTS_FILE.is_file():
        return {}
    points = {p["label"].replace("/", "_").replace(" ", "_"): p
              for front in yaml.safe_load(POINTS_FILE.read_text()).values() for p in front}
    rank = {key: k for k, key in enumerate(points)}
    out = {}
    for name, (_, _, mine) in groups.items():
        found = [(r, points[key]["thresholds"]) for r in mine
                 for key in [f"{r['model']}, {r['setting']}".replace(" ", "_")] if key in points]
        if found:
            out[name] = sorted(found, key=lambda item: rank[f"{item[0]['model']}, {item[0]['setting']}".replace(" ", "_")])
    return out


def candidates_of(rows):
    """The candidates of the test rule among `rows`: the trials of the configured Optuna studies."""
    return [r for r in rows if r.get("configured")]


def points_text(chosen, rule):
    """The chosen points of each group, under the plots."""
    if not chosen:
        return "test points: none (validation: N = 0; test: no row of the points of the rule in the cache yet)"
    n, low, high, metric = rule
    if split_box.value == "test":
        out = io.StringIO()
        print(f"the test rows of the points of the rule ({POINTS_FILE.name}), each run once; numbered as on validation "
              f"(by their APD there): a number out of order on the plot is a point that moved", file=out)
        for name, points in chosen.items():
            print(f"\n{name}: {len(points)} points on the test", file=out)
            print(f"  {'#':>2s} {'APD >=':16s} {'mean':>7s} {'best10':>7s} {'APD':>6s}  {'model':40s} setting", file=out)
            for k, (r, thresholds) in enumerate(points, 1):
                print(f"  {k:2d} {', '.join(f'{a:g}' for a in thresholds):16s} {r['mean']:7.2f} {r['best_of_k']:7.2f} "
                      f"{r['apd']:6.2f}  {r['model'][:40]:40s} {r['setting']}", file=out)
        return out.getvalue()
    same = (n, low, high, metric) == (F.TEST_POINTS, *F.APD_RANGE, "mean")
    out = io.StringIO()
    print(f"test points: {n} thresholds of APD from {low:g} to {high:g} cm; per threshold the lowest "
          f"{METRICS[metric][0].split(' (')[0]} among the trials of the configured studies with APD at least that "
          + ("(the rule of the test, scripts/select_test_points.py)" if same else
             f"(NOT the rule of the test: {F.TEST_POINTS} from {F.APD_RANGE[0]} to {F.APD_RANGE[1]} cm by the mean)"),
          file=out)
    for name, points in chosen.items():
        print(f"\n{name}: {len(points)} points", file=out)
        print(f"  {'#':>2s} {'APD >=':16s} {'mean':>7s} {'best10':>7s} {'APD':>6s}  {'model':26s} setting", file=out)
        for k, (r, thresholds) in enumerate(points, 1):
            print(f"  {k:2d} {', '.join(f'{a:g}' for a in thresholds):16s} {r['mean']:7.2f} {r['best_of_k']:7.2f} "
                  f"{r['apd']:6.2f}  {r['model'][:26]:26s} {r['setting']} ({r['source'].split(':')[-1]}) "
                  f"{short(r['options'])}", file=out)
    return out.getvalue()


def rating(chosen, candidates, rule):
    """{front: (points, hypervolume of its test points, of its whole front)}, the largest first: F.hypervolume over the
    APD band of the rule, against the reference of the metric fixed before the test (F.HV_REFERENCE). The points' staircase equals
    the front's at every threshold and lies above it between them, so the first is at most the second and tends to
    it as N grows."""
    n, low, high, metric = rule
    ref = F.HV_REFERENCE[metric]
    out = {name: (len(points), F.hypervolume([r for r, _ in points], metric, low, high, ref),
                  F.hypervolume(candidates.get(name, []), metric, low, high, ref)) for name, points in chosen.items()}
    return dict(sorted(out.items(), key=lambda kv: -kv[1][1]))


def rating_test_text(chosen, rule):
    """On the test split: the hypervolume of the points of each front on the test, beside that of the same points on
    validation (their numbers in POINTS_FILE); a front is complete when all its points are on the test."""
    import yaml
    n, low, high, metric = rule
    ref, width = F.HV_REFERENCE[metric], high - low
    fronts = yaml.safe_load(POINTS_FILE.read_text())
    validation = {p["label"].replace("/", "_").replace(" ", "_"): p for f in fronts.values() for p in f}
    out = io.StringIO()
    label = METRICS[metric][0].split(" (")[0]
    print(f"\nrating on the test: hypervolume of the points over APD {low:g}-{high:g} cm against {label} {ref} mm "
          f"(mm x cm), and of the same points on validation:", file=out)
    print(f"  {'front':34s} {'points':>9s} {'HV test':>8s} {'HV val':>7s}  {'band mean test':>14s} {'val':>6s}", file=out)
    rows = []
    for name, points in chosen.items():
        keys = [f"{r['model']}, {r['setting']}".replace(" ", "_") for r, _ in points]
        val = [validation[k]["validation"] for k in keys]
        total = next((len(f) for f in fronts.values()
                      if {p["label"].replace("/", "_").replace(" ", "_") for p in f} >= set(keys)), len(keys))
        hv_test = F.hypervolume([r for r, _ in points], metric, low, high)
        hv_val = F.hypervolume(val, metric, low, high)
        rows.append((hv_test, f"  {name:34s} {f'{len(keys)} of {total}':>9s} {hv_test:8.2f} {hv_val:7.2f}  "
                              f"{ref - hv_test / width:14.2f} {ref - hv_val / width:6.2f}"
                              + ("" if len(keys) == total else "  (incomplete)")))
    for _, line in sorted(rows, key=lambda item: -item[0]):
        print(line, file=out)
    return out.getvalue()


def rating_text(chosen, candidates, rule):
    if split_box.value == "test":
        return rating_test_text(chosen, rule) if chosen and POINTS_FILE.is_file() else ""
    scores = rating(chosen, candidates, rule)
    if not scores:
        return ""
    n, low, high, metric = rule
    ref, width = F.HV_REFERENCE[metric], high - low
    label = METRICS[metric][0].split(" (")[0]
    out = io.StringIO()
    print(f"\nrating: hypervolume over APD {low:g}-{high:g} cm against {label} {ref} mm (mm x cm; larger is better), "
          f"and the band mean it equals (ref - hypervolume / {width:g}); validation:", file=out)
    print(f"  {'front':34s} {'points':>6s} {'HV points':>10s} {'HV front':>9s} {'lost':>6s}  "
          f"{'band mean points':>16s} {'front':>6s}", file=out)
    for name, (k, points, whole) in scores.items():
        print(f"  {name:34s} {k:6d} {points:10.2f} {whole:9.2f} {whole - points:6.2f}  {ref - points / width:16.2f} "
              f"{ref - whole / width:6.2f}", file=out)
    print("  HV points: of the N test points (what the test can measure); HV front: of all trials of the configured\n"
          "  studies; lost: what the split into N thresholds leaves out. A fixed reference: adding a front changes no\n"
          "  other's value.", file=out)
    return out.getvalue()


def spec_points(chosen):
    """The chosen points as the file of scripts/select_test_points.py ({front: [point]}, for evaluate_completion.py
    --points): the spec of each from the configuration its study recorded."""
    import select_test_points as S
    out, studies = {}, {}
    for name, points in chosen.items():
        out[name] = []
        for r, thresholds in points:
            path, study_name = r["source"].split(":")
            key = (path, study_name)
            if key not in studies:
                studies[key] = optuna.load_study(study_name=study_name,
                                                 storage=f"sqlite:///{RESULTS / 'optuna' / path}")
            study = studies[key]
            number = int(r["setting"].split()[-1])
            config = study.user_attrs["config"]
            point = dict(study=study_name, trial=number, spec=S.spec_of(config, study.trials[number].params))
            out[name].append(dict(label=S.label_of(S.family_of(config["model"]), point), study=study_name,
                                  trial=number, thresholds=thresholds,
                                  validation={k: round(r[k], 3) for k in ("mean", "best_of_k", "apd")},
                                  spec=point["spec"]))
    return out


def figure(rows, models, occlusion, style, model_fronts, labelled, split="validation", by="family", points=True,
           rule=(0, *F.APD_RANGE, "mean"), numbers=False, values=False):
    """The plots of PANELS for the records `rows` (of one split) of the models `models`, a front per group of `by`
    ("family", or "family x method", each in its colour of GROUP_COLORS): faint small points of all experiments
    (points=False: none) and the front of each group as a line: solid for ours (OURS), dashed for the others; the
    test points of each group by `rule` (test_points) as crosses, optionally with their number in the front (in the
    order of APD, as in the list under the plots) and their values (APD, y). Returns the figure, the points for clicks and the
    test points."""
    rows = [r for r in at(rows, occlusion) if r["model"] in models]
    groups = groups_of(rows, by)
    chosen = test_points(groups, rule, split)
    candidates = {name: candidates_of(mine) for name, (_, _, mine) in groups.items()}
    drawn = []
    with plt.style.context("default"):
        fig, axes = plt.subplots(1, len(PANELS), figsize=(7.5 * len(PANELS), 6.5))
        axes = np.atleast_1d(axes)
        labels = {ax: [] for ax in axes}
        for ax, (x, y) in zip(axes, PANELS):
            seen = [r for r in rows if all(r[m] <= LIMITS.get(m, np.inf) for m in (x, y))]
            for name, (family, method, mine) in groups.items():
                color = GROUP_COLORS.get((family, method), FAMILIES[family])
                for r in (r for r in mine if r in seen):
                    artist, = ax.plot(r[x], r[y], marker="o", ls="", ms=5, alpha=0.2 if points else 0, color=color,
                                      mec="none", zorder=1)
                    drawn.append((artist, r[x], r[y], r["model"]))
                if model_fronts:
                    for model in sorted({r["model"] for r in mine}):
                        front = F.front([r for r in mine if r["model"] == model], *objectives(x, y))
                        if len(front) > 1:
                            V.draw_front(ax, [r[x] for r in front], [r[y] for r in front], style, color=color,
                                         ls="--", lw=0.7, ring=6, corner=CORNER)
                front = F.front(mine, *objectives(x, y))
                V.draw_front(ax, [r[x] for r in front], [r[y] for r in front], style, color=color,
                             ls="-" if family == OURS else "--", lw=1.8, ring=8, corner=CORNER)
                marked = [r for r, _ in chosen.get(name, []) if r in seen]
                ax.plot([r[x] for r in marked], [r[y] for r in marked], marker="x", ls="", ms=10, mew=2.2,
                        color=color, zorder=6)
                if numbers or values:
                    order = {id(r): k for k, (r, _) in enumerate(chosen.get(name, []), 1)}
                    for r in marked:
                        text = " ".join(filter(None, [f"{order[id(r)]}" if numbers else "",
                                                      f"({r[x]:.2f}, {r[y]:.1f})" if values else ""]))
                        ax.annotate(text, (r[x], r[y]), xytext=(5, -9), textcoords="offset points", fontsize=7,
                                    color=color, zorder=7)
                if labelled:
                    labels[ax] += [(r[x], r[y], f"{r['model']}, {SHORT[r['method']]}: {r['setting']}")
                                   for r in front if r in seen]
            if occlusion is None and split == "validation" and "apd" in (x, y):
                (ax.axvline if x == "apd" else ax.axhline)(APD_FLOOR, color="0.5", lw=1, ls=":", zorder=0)
            if seen and len(seen) < len(rows):
                for set_lim, coords in ((ax.set_xlim, [r[x] for r in seen]), (ax.set_ylim, [r[y] for r in seen])):
                    low, high = min(coords), max(coords)
                    set_lim(low - 0.04 * (high - low), high + 0.04 * (high - low))
                beyond = ", ".join(f"{METRICS[m][0].split(' (')[0]} {LIMITS[m]} mm" for m in (x, y) if m in LIMITS)
                ax.set_title(f"{len(rows) - len(seen)} experiments beyond {beyond}: not drawn", fontsize=8.5,
                             color="0.35")
            ax.set_xlabel(f"{METRICS[x][0]}   {arrow(x, 'x')}")
            ax.set_ylabel(f"{METRICS[y][0]}   {arrow(y, 'y')}")
            ax.grid(True, alpha=0.3)
        handles = [plt.Line2D([], [], color=GROUP_COLORS.get((family, method), FAMILIES[family]), lw=1.8,
                              ls="-" if family == OURS else "--", label=name)
                   for name, (family, method, mine) in groups.items()]
        if occlusion is None and split == "validation":
            handles += [plt.Line2D([], [], color="0.5", ls=":", label=f"APD of RSGM D-Flow Heun EDM 50 ({APD_FLOOR} cm)")]
        if chosen:
            handles += [plt.Line2D([], [], color="0.3", marker="x", ls="", ms=8, mew=2,
                                   label=("the points of the rule, on the test" if split == "test" else
                                          f"test points ({rule[0]} thresholds of APD {rule[1]:g}-{rule[2]:g} cm, "
                                          f"lowest {METRICS[rule[3]][0].split(' (')[0]})"))]
        axes[0].legend(handles=handles, loc=LEGEND, fontsize=8)
        poses = rows[0]["poses"] if rows else "no"
        where = "mean over the occlusions" if occlusion is None else occlusion
        fig.suptitle(f"Pose completion on {poses} {split} poses, K = 10, {where}", fontsize=12)
        fig.tight_layout(rect=(0, 0, 1, 0.96))
        fig.patch.set_facecolor("white")
        if labelled:
            V.place_labels(fig, labels, fontsize=6.5)
    height = fig.bbox.height
    hits = [(a, p[0], height - p[1], m) for a, x, y, m in drawn for p in [a.axes.transData.transform((x, y))]]
    return fig, hits, chosen, candidates


# ---- the controls ----
L = lambda w: widgets.Layout(width=w)
split_box = widgets.ToggleButtons(options=["validation", "test"], value=SPLIT, description="split",
                                  layout=L("330px"), style={"button_width": "100px"})
poses_box = widgets.Dropdown(options=sorted({r["poses"] for r in records if r["split"] == SPLIT}),
                             value=POSES if SPLIT == "validation" else None, description="poses", layout=L("170px"))
if poses_box.value is None:
    poses_box.value = poses_box.options[-1]
occlusion_box = widgets.Dropdown(options=[("mean of the four", "mean")] + [(o, o) for o in OCCLUSIONS],
                                 description="occlusion", layout=L("260px"))
style_box = widgets.Dropdown(options=[(d, s) for s, d in V.STYLES.items()], value=FRONT_STYLE, description="fronts",
                             layout=L("360px"))
model_fronts_box = widgets.Checkbox(value=MODEL_FRONTS, description="a front per model", indent=False, layout=L("170px"))
labels_box = widgets.Checkbox(value=LABELS, description="labels", indent=False, layout=L("100px"))
points_box = widgets.Checkbox(value=POINTS, description="all experiments (else only the fronts)", indent=False,
                              layout=L("260px"))
points_n_box = widgets.IntSlider(value=TEST_POINTS, min=0, max=20, description="test points per front",
                                 layout=L("360px"), style={"description_width": "initial"})
apd_range_box = widgets.FloatRangeSlider(value=F.APD_RANGE, min=2.0, max=12.0, step=0.1, description="APD from, to (cm)",
                                         layout=L("420px"), style={"description_width": "initial"})
numbers_box = widgets.Checkbox(value=False, description="validation numbering", indent=False, layout=L("170px"))
values_box = widgets.Checkbox(value=False, description="values of the points", indent=False, layout=L("170px"))
point_metric_box = widgets.Dropdown(options=[("lowest mean", "mean"), ("lowest best of 10", "best_of_k")], value="mean",
                                    description="by", layout=L("220px"))
groups_box = widgets.ToggleButtons(options=["family", "family x method"], value=GROUPS, description="a front per",
                                   layout=L("420px"), style={"button_width": "130px"})
model_boxes = widgets.Accordion()  # a collapsible section of checkboxes per family
path_field = widgets.Text(description="save to", layout=L("700px"), style={"description_width": "initial"})
save_button = widgets.Button(description="save", icon="save", layout=L("90px"))
status, picked = widgets.Label(), widgets.Label()
points_path = widgets.Text(value=str(RESULTS / "test_points_notebook.yaml"), description="points to",
                           layout=L("700px"), style={"description_width": "initial"})
points_button = widgets.Button(description="save points", icon="save", layout=L("120px"))
points_status = widgets.Label()
image = widgets.Image(format="png", layout=widgets.Layout(width="100%"))
text = widgets.HTML()
state = {"fig": None, "hits": [], "rings": [], "highlight": None, "default": None}


def rule():
    return (points_n_box.value, *(round(v, 2) for v in apd_range_box.value), point_metric_box.value)


def current():
    return [r for r in records if r["split"] == split_box.value and r["poses"] == poses_box.value]


def set_split(*_):
    """The poses of the chosen split (the 1000 of the studies if there, else the most), then the models."""
    options = sorted({r["poses"] for r in records if r["split"] == split_box.value})
    value = POSES if POSES in options else options[-1]
    poses_box.unobserve(set_choices, names="value")
    poses_box.options = options
    poses_box.value = value
    poses_box.observe(set_choices, names="value")
    set_choices()


def checked():
    return [box.model for section in model_boxes.children for box in section.children[1:] if box.value]


def set_choices(*_):
    """A collapsible section of checkboxes per family: the models of the chosen poses (all checked; the first box
    checks or clears all of the family), then draw."""
    rows = current()
    sections, titles = [], []
    for family in FAMILIES:
        boxes = []
        for model in MODELS:
            if family_of(model) == family and any(r["model"] == model for r in rows):
                n = sum(r["model"] == model for r in rows)
                box = widgets.Checkbox(value=True, description=f"{model} ({n})", indent=False, layout=L("330px"))
                box.model = model
                box.observe(redraw, names="value")
                boxes.append(box)
        if boxes:
            every = widgets.Checkbox(value=True, description="all", indent=False, layout=L("330px"))
            every.observe(lambda change, boxes=boxes: check_all(boxes, change["new"]), names="value")
            sections.append(widgets.GridBox([every] + boxes, layout=widgets.Layout(
                grid_template_columns="repeat(3, 340px)")))
            titles.append(f"{family}: {len(boxes)} models ({sum(r['family'] == family for r in rows)} experiments)")
    model_boxes.children = sections
    for k, title in enumerate(titles):
        model_boxes.set_title(k, title)
    redraw()


def check_all(boxes, value):
    """Check or clear the models of a family, then draw once."""
    for box in boxes:
        box.unobserve(redraw, names="value")
        box.value = value
        box.observe(redraw, names="value")
    redraw()


def redraw(*_):
    try:
        draw()
    except Exception:
        import traceback
        text.value = f"<pre style='color: #b00'>{html.escape(traceback.format_exc())}</pre>"


def draw():
    models = checked()
    occlusion = None if occlusion_box.value == "mean" else occlusion_box.value
    tag = "v" if split_box.value == "validation" else "test"
    default = str(FIGURES / f"completion_fronts_{tag}{poses_box.value}_{occlusion or 'mean'}.png")
    if path_field.value in ("", state["default"]):
        path_field.value = state["default"] = default
    if not models:
        image.value, text.value, state["fig"] = b"", "<pre>no model checked</pre>", None
        return
    fig, hits, chosen, candidates = figure(current(), models, occlusion, style_box.value, model_fronts_box.value, labels_box.value,
                                           split_box.value, groups_box.value, points_box.value, rule(),
                                           numbers_box.value, values_box.value)
    plt.close(fig)
    state.update(fig=fig, hits=hits, rings=[])
    show()
    state["chosen"] = chosen
    text.value = (f"<pre style='font-size: 12px; line-height: 1.3'>"
                  f"{html.escape(points_text(chosen, rule()) + rating_text(chosen, candidates, rule()))}</pre>")


def show():
    for ring in state["rings"]:
        ring.remove()
    if state["highlight"] not in {m for *_, m in state["hits"]}:
        state["highlight"] = None
    state["rings"] = V.highlight(state["hits"], state["highlight"])
    picture = io.BytesIO()
    state["fig"].savefig(picture, format="png", dpi=100, facecolor="white")
    image.value = picture.getvalue()
    picked.value = f"highlighted: {state['highlight']}" if state["highlight"] else ""


def save_points(_):
    """The test points drawn now, as a file for scripts/evaluate_completion.py --points (YAML)."""
    if not state.get("chosen"):
        points_status.value = "no test points (validation split, N > 0)"
        return
    import yaml
    n, low, high, metric = rule()
    target = Path(points_path.value).expanduser()
    target.parent.mkdir(parents=True, exist_ok=True)
    header = (f"# Test points of completion_fronts.ipynb: {n} thresholds of APD from {low:g} to {high:g} cm, the lowest "
              f"{metric}; occlusion {occlusion_box.value}, fronts per {groups_box.value}, {len(checked())} models.\n"
              f"# For scripts/evaluate_completion.py --points; the rule of the test is config/completion_test_points.yaml.\n")
    target.write_text(header + yaml.safe_dump(spec_points(state["chosen"]), sort_keys=False, width=120))
    points_status.value = f"saved {target}"


def save(_):
    if state["fig"] is None:
        status.value = "nothing to save"
        return
    target = Path(path_field.value).expanduser()
    target.parent.mkdir(parents=True, exist_ok=True)
    state["fig"].savefig(target, dpi=200, facecolor="white", bbox_inches="tight")
    status.value = f"saved {target}"


def on_click(event):
    x, y = event.get("dataX"), event.get("dataY")
    if x is None or not state["hits"]:
        return
    scale = 100 / state["fig"].dpi
    distance, model = min((np.hypot(hx * scale - x, hy * scale - y), m) for _, hx, hy, m in state["hits"])
    state["highlight"] = model if distance < 12 and model != state["highlight"] else None
    show()


try:
    from ipyevents import Event
    CLICKS = Event(source=image, watched_events=["click"])
    CLICKS.on_dom_event(on_click)
except ImportError:
    CLICKS = None

split_box.observe(set_split, names="value")
poses_box.observe(set_choices, names="value")
for box in (occlusion_box, style_box, model_fronts_box, labels_box, groups_box, points_box, points_n_box, apd_range_box,
            point_metric_box, numbers_box, values_box):
    box.observe(redraw, names="value")
save_button.on_click(save)
points_button.on_click(save_points)
display(widgets.VBox([widgets.HBox([split_box, poses_box, occlusion_box]), widgets.HBox([groups_box, style_box]), widgets.HBox([points_box, model_fronts_box, labels_box]),
                      widgets.HBox([points_n_box, apd_range_box, point_metric_box]), widgets.HBox([numbers_box, values_box]),
                      model_boxes, widgets.HBox([path_field, save_button, status]),
                      widgets.HBox([points_path, points_button, points_status]), picked, image, text]))
set_choices()

## 3a. How many test points per front

The test can measure a front only through its points, so the hypervolume of N points (`hp.evaluation.fronts.hypervolume`,
APD 4.0-8.5 cm against mean 115 mm) is compared with that of the whole front (all trials of the configured studies),
on validation, for every N, with the cost of the test of the new points (5 times the minutes of their validation trial;
PoseD-Flow left out of the cost while its study runs). The rule of the test takes N = 10: at least 96 % of every front
(a loss of 0.1-0.8 mm in the band mean), the same order of the fronts as the whole fronts, a step of 0.5 cm, about 16
GPU hours. The curve is not monotone (the grids of thresholds of different N are not nested), so N is read off the
level, not the first N to cross it.


In [ ]:
import select_test_points as S
from evaluate_completion import MODELS as TEST_ROWS

fronts = S.candidates(RESULTS / "optuna")  # {(family, method): candidates}, as the rule of the test
names = sorted(fronts)
whole = {k: F.hypervolume(fronts[k]) for k in names}
print("% of the hypervolume of the whole front kept by the N test points (mean, APD 4.0-8.5 cm, against 115 mm)")
print(f"{'N':>3s} " + " ".join(f"{f[:10] + ' ' + m:>17s}" for f, m in names) + f" {'min %':>6s} {'points':>6s} {'new':>4s} {'GPU h':>6s}")
for n in range(2, 21):
    kept, points, new, hours = [], 0, 0, 0.0
    for key in names:
        chosen = F.spanning(fronts[key], n)
        kept.append(100 * F.hypervolume([p for p, _ in chosen]) / whole[key])
        points += len(chosen)
        for p, _ in chosen:
            if key[0] != "PoseD-Flow" and S.label_of(key[0], p) not in TEST_ROWS:
                new, hours = new + 1, hours + 5 * (p["minutes"] or 0) / 60
    mark = "  <- the rule" if n == F.TEST_POINTS else ""
    print(f"{n:3d} " + " ".join(f"{k:17.1f}" for k in kept) + f" {min(kept):6.1f} {points:6d} {new:4d} {hours:6.1f}{mark}")


## 4. Reading

To be written with the results of the studies on the 1000 validation poses (`rsgm_inpaint_pareto`,
`rsgm_inpaint_mean_apd_*`): copy their `.db` files from the servers into `optuna/` under `RESULTS`.